# Content-Based Recommendation — Full Catalog

Pipeline:

FULL product metadata
→ Hashing TF-IDF
→ sparse item vectors
→ user profile
→ cosine similarity
→ Top-K recommendation

Quy ước:

- Metadata candidate: TRAIN + VAL + TEST catalog.
- VAL user history: TRAIN interactions.
- TEST user history: TRAIN + VAL interactions.
- TEST interactions chỉ dùng làm ground truth.
- Các bước dài đều có checkpoint/resume.

In [26]:
from pathlib import Path
import sys
import gc
import json
import math

import duckdb
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize

from IPython.display import display


# ============================================================
# PROJECT
# ============================================================

PROJECT_ROOT = Path(r"D:\MerRec")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# ============================================================
# PATHS
# ============================================================

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "recommender"
)

MODEL_DATA_DIR = (
    PROCESSED_ROOT
    / "model_data"
)

SERVING_DIR = (
    PROCESSED_ROOT
    / "serving"
)

INTERACTIONS_DIR = (
    MODEL_DATA_DIR
    / "interactions"
)


TRAIN_GLOB = (
    INTERACTIONS_DIR
    / "split=train"
    / "*.parquet"
).as_posix()

VAL_GLOB = (
    INTERACTIONS_DIR
    / "split=val"
    / "*.parquet"
).as_posix()

TEST_GLOB = (
    INTERACTIONS_DIR
    / "split=test"
    / "*.parquet"
).as_posix()


# FULL METADATA ĐÃ ĐƯỢC TẠO Ở PREPROCESSING
CATALOG_SOURCE = (
    SERVING_DIR
    / "item_catalog_full.parquet"
)


# ============================================================
# MODEL OUTPUT
# ============================================================

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "training"
    / "checkpoints"
)

OUT = (
    CHECKPOINT_DIR
    / "content_based_full_catalog"
)

SHARD_DIR = (
    OUT
    / "matrix_shards"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

SHARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("PROJECT :", PROJECT_ROOT)
print("CATALOG :", CATALOG_SOURCE)
print("OUTPUT  :", OUT)

PROJECT : D:\MerRec
CATALOG : D:\MerRec\data\processed\recommender\serving\item_catalog_full.parquet
OUTPUT  : D:\MerRec\training\checkpoints\content_based_full_catalog


In [27]:
# ============================================================
# CONTENT-BASED CONFIG
# ============================================================

BATCH_SIZE = 100_000


# Hash space
HASH_FEATURES = 2 ** 18
# = 262,144 text dimensions


PRICE_WEIGHT = 0.20


# ============================================================
# USER PROFILE
# ============================================================

PROFILE_MAX_ITEMS = 100

RECENCY_DECAY = 0.97


# ============================================================
# EVALUATION
# ============================================================

EVAL_USERS_VAL = 500
EVAL_USERS_TEST = 500

K_LIST = [10, 20]

QUERY_BATCH = 8


# ============================================================
# CHECKPOINTS
# ============================================================

HASHER_PATH = (
    OUT
    / "hashing_vectorizer.joblib"
)

IDF_PATH = (
    OUT
    / "idf.npy"
)

DF_CHECKPOINT = (
    OUT
    / "idf_df_counts.npy"
)

STATE_CHECKPOINT = (
    OUT
    / "idf_state.json"
)

META_PATH = (
    OUT
    / "meta.json"
)

MANIFEST_PATH = (
    OUT
    / "manifest.json"
)

VAL_RESULT_PATH = (
    OUT
    / "evaluation_val.csv"
)

TEST_RESULT_PATH = (
    OUT
    / "evaluation_test.csv"
)


print(
    "HASH_FEATURES:",
    f"{HASH_FEATURES:,}"
)

print(
    "BATCH_SIZE:",
    f"{BATCH_SIZE:,}"
)

HASH_FEATURES: 262,144
BATCH_SIZE: 100,000


In [28]:
# ============================================================
# INPUT VALIDATION
# ============================================================

if not CATALOG_SOURCE.exists():

    raise FileNotFoundError(
        f"Không thấy FULL catalog:\n"
        f"{CATALOG_SOURCE}"
    )


if CATALOG_SOURCE.stat().st_size <= 0:

    raise RuntimeError(
        f"FULL catalog rỗng:\n"
        f"{CATALOG_SOURCE}"
    )


con = duckdb.connect()


# ============================================================
# CATALOG SCHEMA
# ============================================================

schema = con.execute(f"""
    DESCRIBE

    SELECT *

    FROM read_parquet(
        '{CATALOG_SOURCE.as_posix()}'
    )
""").df()


required_catalog_cols = {
    "item_id",
    "name",
    "price",
    "category0",
    "category1",
    "category2",
    "brand",
    "condition",
}


catalog_cols = set(
    schema["column_name"]
    .astype(str)
)


missing = sorted(
    required_catalog_cols
    - catalog_cols
)


if missing:

    con.close()

    raise ValueError(
        f"FULL catalog thiếu columns: "
        f"{missing}"
    )


# ============================================================
# CATALOG STATS
# ============================================================

catalog_stats = con.execute(f"""
    SELECT

        COUNT(*) AS items,

        COUNT(
            DISTINCT item_id
        ) AS distinct_items

    FROM read_parquet(
        '{CATALOG_SOURCE.as_posix()}'
    )
""").df()


# ============================================================
# INTERACTION SCHEMA
# ============================================================

interaction_schema = con.execute(f"""
    DESCRIBE

    SELECT *

    FROM read_parquet(
        '{TRAIN_GLOB}'
    )
""").df()


con.close()


interaction_cols = set(
    interaction_schema[
        "column_name"
    ].astype(str)
)


required_interaction_cols = {
    "user_id",
    "item_id",
    "event_group",
    "event_weight",
    "ts",
}


missing_interactions = sorted(
    required_interaction_cols
    - interaction_cols
)


if missing_interactions:

    raise ValueError(
        "Interaction dataset thiếu columns: "
        f"{missing_interactions}"
    )


FULL_ITEMS = int(
    catalog_stats.loc[
        0,
        "items"
    ]
)

DISTINCT_ITEMS = int(
    catalog_stats.loc[
        0,
        "distinct_items"
    ]
)


display(
    catalog_stats
)


if FULL_ITEMS != DISTINCT_ITEMS:

    raise RuntimeError(
        "FULL catalog phải có "
        "đúng 1 row / item_id"
    )


print()
print("✅ INPUT OK")

print(
    "FULL ITEMS:",
    f"{FULL_ITEMS:,}"
)

,items,distinct_items
0,30044194,30044194



✅ INPUT OK
FULL ITEMS: 30,044,194


In [ ]:
# ============================================================
# TEXT PREPROCESSING
# ============================================================

def token_value(
    value,
    prefix
):

    if pd.isna(value):

        value = "__UNK__"


    value = (
        str(value)
        .strip()
        .lower()
    )


    value = "_".join(
        value.split()
    )


    return (
        f"{prefix}_{value}"
    )


def build_text(df):

    name = (
        df["name"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    c0 = df["category0"].map(
        lambda x:
        token_value(
            x,
            "c0"
        )
    )


    c1 = df["category1"].map(
        lambda x:
        token_value(
            x,
            "c1"
        )
    )


    c2 = df["category2"].map(
        lambda x:
        token_value(
            x,
            "c2"
        )
    )


    brand = df["brand"].map(
        lambda x:
        token_value(
            x,
            "brand"
        )
    )


    condition = df["condition"].map(
        lambda x:
        token_value(
            x,
            "condition"
        )
    )


    return (
        name
        + " "
        + c0
        + " "
        + c1
        + " "
        + c2
        + " "
        + brand
        + " "
        + condition
    )

In [29]:
# Cell 4 — Input validation ONLY
# Không xóa checkpoint
# Không xóa matrix shard
# Không migrate tự động

if not CATALOG_SOURCE.exists():
    raise FileNotFoundError(
        f"Không thấy full catalog:\n{CATALOG_SOURCE}"
    )

if CATALOG_SOURCE.stat().st_size <= 0:
    raise RuntimeError(
        f"Full catalog bị rỗng:\n{CATALOG_SOURCE}"
    )


con = duckdb.connect()

schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{CATALOG_SOURCE.as_posix()}'
    )
""").df()


required = {
    "item_id",
    "name",
    "price",
    "category0",
    "category1",
    "category2",
    "brand",
    "condition",
}

cols = set(
    schema["column_name"]
    .astype(str)
)

missing = sorted(
    required - cols
)

if missing:
    con.close()

    raise ValueError(
        f"Full catalog thiếu columns: {missing}"
    )


catalog_stats = con.execute(f"""
    SELECT
        COUNT(*) AS items,
        COUNT(DISTINCT item_id) AS distinct_items

    FROM read_parquet(
        '{CATALOG_SOURCE.as_posix()}'
    )
""").df()


con.close()


FULL_ITEMS = int(
    catalog_stats.loc[0, "items"]
)

DISTINCT_ITEMS = int(
    catalog_stats.loc[0, "distinct_items"]
)


display(catalog_stats)


if FULL_ITEMS != DISTINCT_ITEMS:
    raise RuntimeError(
        "Full catalog phải có đúng 1 row / item_id."
    )


print()
print("✅ Input OK")
print("Catalog :", CATALOG_SOURCE)
print("Items   :", f"{FULL_ITEMS:,}")

,items,distinct_items
0,30044194,30044194



✅ Input OK
Catalog : D:\MerRec\data\processed\recommender\serving\item_catalog_full.parquet
Items   : 30,044,194


In [30]:
# ============================================================
# PRICE NORMALIZATION
# FULL CATALOG
# ============================================================

con = duckdb.connect()


price_stats = con.execute(f"""
    SELECT

        AVG(
            LN(
                1
                +
                GREATEST(
                    COALESCE(
                        price,
                        0
                    ),
                    0
                )
            )
        ) AS price_log_mean,

        STDDEV_POP(
            LN(
                1
                +
                GREATEST(
                    COALESCE(
                        price,
                        0
                    ),
                    0
                )
            )
        ) AS price_log_std

    FROM read_parquet(
        '{CATALOG_SOURCE.as_posix()}'
    )
""").df()


con.close()


PRICE_LOG_MEAN = float(
    price_stats.loc[
        0,
        "price_log_mean"
    ]
)


PRICE_LOG_STD = float(
    price_stats.loc[
        0,
        "price_log_std"
    ]
)


if (
    not np.isfinite(
        PRICE_LOG_STD
    )
    or PRICE_LOG_STD < 1e-6
):

    PRICE_LOG_STD = 1.0


print(
    "PRICE_LOG_MEAN:",
    PRICE_LOG_MEAN
)

print(
    "PRICE_LOG_STD:",
    PRICE_LOG_STD
)

PRICE_LOG_MEAN: 3.16638053189261
PRICE_LOG_STD: 0.9561930811493935


In [31]:
# ============================================================
# HASHING TF-IDF
# STREAMING + CHECKPOINT + RESUME
# ============================================================


def make_hasher():

    return HashingVectorizer(

        n_features=HASH_FEATURES,

        ngram_range=(
            1,
            2
        ),

        alternate_sign=False,

        norm=None,

        dtype=np.float32,

        strip_accents="unicode",
    )


# ============================================================
# HASHER
# ============================================================

hasher = make_hasher()


# ============================================================
# FINAL ARTIFACT ĐÃ CÓ
# ============================================================

if (
    HASHER_PATH.exists()
    and
    IDF_PATH.exists()
):

    print(
        "✅ Hashing TF-IDF "
        "đã hoàn thành -> REUSE"
    )


    hasher = joblib.load(
        HASHER_PATH
    )


    IDF = np.load(
        IDF_PATH,
        mmap_mode="r"
    )


# ============================================================
# CHƯA HOÀN THÀNH
# ============================================================

else:

    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if (
        DF_CHECKPOINT.exists()
        and
        STATE_CHECKPOINT.exists()
    ):

        print(
            "♻️ Tìm thấy IDF checkpoint"
        )


        df_counts = np.load(
            DF_CHECKPOINT
        )


        state = json.loads(
            STATE_CHECKPOINT.read_text(
                encoding="utf-8"
            )
        )


        n_documents = int(
            state[
                "n_documents"
            ]
        )


        saved_features = int(
            state[
                "hash_features"
            ]
        )


        if (
            saved_features
            != HASH_FEATURES
        ):

            raise RuntimeError(
                "Hash feature config "
                "không khớp checkpoint."
            )


        print(
            "Resume từ:",
            f"{n_documents:,}"
            "/",
            f"{FULL_ITEMS:,}"
        )


    else:

        print(
            "🚀 IDF bắt đầu từ 0"
        )


        df_counts = np.zeros(
            HASH_FEATURES,
            dtype=np.int64
        )


        n_documents = 0


    # --------------------------------------------------------
    # STREAM CATALOG
    # --------------------------------------------------------

    con = duckdb.connect()


    reader = con.execute(f"""
        SELECT

            name,
            category0,
            category1,
            category2,
            brand,
            condition

        FROM read_parquet(
            '{CATALOG_SOURCE.as_posix()}'
        )

        LIMIT 9223372036854775807

        OFFSET {n_documents}
    """).fetch_record_batch(
        BATCH_SIZE
    )


    for batch in reader:

        df_batch = (
            batch.to_pandas()
        )


        texts = build_text(
            df_batch
        )


        # ====================================================
        # HASH TERM FREQUENCY
        # ====================================================

        X = hasher.transform(
            texts
        ).tocsr()


        # ====================================================
        # DOCUMENT FREQUENCY
        #
        # CSR mỗi document-feature chỉ có 1 entry.
        # Đưa mọi non-zero về 1.
        # ====================================================

        if X.nnz > 0:

            X.data[:] = 1.0


        batch_df = np.asarray(
            X.sum(
                axis=0
            )
        ).ravel()


        df_counts += (
            batch_df.astype(
                np.int64
            )
        )


        n_documents += len(
            df_batch
        )


        # ====================================================
        # CHECKPOINT SAU MỖI BATCH
        # ====================================================

        tmp_df_path = (
            OUT
            / "idf_df_counts_tmp.npy"
        )

        tmp_state_path = (
            OUT
            / "idf_state_tmp.json"
        )


        np.save(
            tmp_df_path,
            df_counts
        )


        tmp_state_path.write_text(
            json.dumps(
                {
                    "n_documents":
                        n_documents,

                    "total_items":
                        FULL_ITEMS,

                    "hash_features":
                        HASH_FEATURES,
                },
                indent=2
            ),
            encoding="utf-8"
        )


        tmp_df_path.replace(
            DF_CHECKPOINT
        )


        tmp_state_path.replace(
            STATE_CHECKPOINT
        )


        print(
            "💾 IDF:",
            f"{n_documents:,}"
            "/",
            f"{FULL_ITEMS:,}"
        )


        del (
            df_batch,
            texts,
            X,
            batch_df
        )


        gc.collect()


    con.close()


    # ========================================================
    # COMPUTE FINAL IDF
    # ========================================================

    IDF = (

        np.log(

            (
                1.0
                +
                n_documents
            )

            /

            (
                1.0
                +
                df_counts
            )

        )

        + 1.0

    ).astype(
        np.float32
    )


    # ========================================================
    # SAVE FINAL
    # ========================================================

    np.save(
        IDF_PATH,
        IDF
    )


    joblib.dump(
        hasher,
        HASHER_PATH,
        compress=3
    )


    # ========================================================
    # CHỈ XÓA TEMP CHECKPOINT KHI FINAL ĐÃ SAVE XONG
    # ========================================================

    if DF_CHECKPOINT.exists():

        DF_CHECKPOINT.unlink()


    if STATE_CHECKPOINT.exists():

        STATE_CHECKPOINT.unlink()


    IDF = np.load(
        IDF_PATH,
        mmap_mode="r"
    )


    print()
    print(
        "✅ HASHING TF-IDF COMPLETE"
    )

    print(
        "Documents:",
        f"{n_documents:,}"
    )

    print(
        "Features:",
        f"{HASH_FEATURES:,}"
    )

🚀 IDF bắt đầu từ 0


C:\Users\ASUS\AppData\Local\Temp\ipykernel_5556\4121147112.py:168: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(


💾 IDF: 100,000/ 30,044,194
💾 IDF: 200,000/ 30,044,194
💾 IDF: 300,000/ 30,044,194
💾 IDF: 400,000/ 30,044,194
💾 IDF: 500,000/ 30,044,194
💾 IDF: 600,000/ 30,044,194
💾 IDF: 700,000/ 30,044,194
💾 IDF: 800,000/ 30,044,194
💾 IDF: 900,000/ 30,044,194
💾 IDF: 1,000,000/ 30,044,194
💾 IDF: 1,100,000/ 30,044,194
💾 IDF: 1,200,000/ 30,044,194
💾 IDF: 1,300,000/ 30,044,194
💾 IDF: 1,400,000/ 30,044,194
💾 IDF: 1,500,000/ 30,044,194
💾 IDF: 1,600,000/ 30,044,194
💾 IDF: 1,700,000/ 30,044,194
💾 IDF: 1,800,000/ 30,044,194
💾 IDF: 1,900,000/ 30,044,194
💾 IDF: 2,000,000/ 30,044,194
💾 IDF: 2,100,000/ 30,044,194
💾 IDF: 2,200,000/ 30,044,194
💾 IDF: 2,300,000/ 30,044,194
💾 IDF: 2,400,000/ 30,044,194
💾 IDF: 2,500,000/ 30,044,194
💾 IDF: 2,600,000/ 30,044,194
💾 IDF: 2,700,000/ 30,044,194
💾 IDF: 2,800,000/ 30,044,194
💾 IDF: 2,900,000/ 30,044,194
💾 IDF: 3,000,000/ 30,044,194
💾 IDF: 3,100,000/ 30,044,194
💾 IDF: 3,200,000/ 30,044,194
💾 IDF: 3,300,000/ 30,044,194
💾 IDF: 3,400,000/ 30,044,194
💾 IDF: 3,500,000/ 30,044,194
💾 I

In [32]:
# ============================================================
# METADATA -> HASHING TF-IDF + PRICE
# ============================================================


# Tạo object mới sạch.
# HashingVectorizer KHÔNG cần fit().
hash_vectorizer = make_hasher()


# ============================================================
# LOAD IDF
# ============================================================

if not IDF_PATH.exists():

    raise RuntimeError(
        "Chưa có idf.npy. "
        "Chạy Cell 7 hoàn thành trước."
    )


IDF = np.load(
    IDF_PATH,
    mmap_mode="r"
)


if IDF.shape[0] != HASH_FEATURES:

    raise RuntimeError(
        f"IDF dimension sai: "
        f"{IDF.shape[0]:,} "
        f"!= "
        f"{HASH_FEATURES:,}"
    )


print(
    "✅ HashingVectorizer ready"
)

print(
    "✅ IDF ready:",
    IDF.shape
)


# ============================================================
# TRANSFORM FUNCTION
# ============================================================

def transform_metadata(df):

    texts = build_text(
        df
    )


    # ========================================================
    # 1. HASH TERM FREQUENCY
    # ========================================================

    X = (
        hash_vectorizer
        .transform(
            texts
        )
        .tocsr()
    )


    # ========================================================
    # 2. MANUAL SUBLINEAR TF
    #
    # tf -> 1 + log(tf)
    # ========================================================

    if X.nnz > 0:

        X.data = (
            1.0
            +
            np.log(
                X.data
            )
        ).astype(
            np.float32
        )


    # ========================================================
    # 3. TF × IDF
    # ========================================================

    X = (
        X.multiply(
            IDF
        )
        .tocsr()
    )


    # ========================================================
    # 4. PRICE
    # ========================================================

    price = (
        df["price"]
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )


    price = np.maximum(
        price,
        0
    )


    price_log = np.log1p(
        price
    )


    price_z = (

        (
            price_log
            -
            PRICE_LOG_MEAN
        )

        /

        PRICE_LOG_STD

    ).astype(
        np.float32
    )


    # ========================================================
    # 5. CONCAT TEXT + PRICE
    # ========================================================

    X = sp.hstack(
        [
            X,

            sp.csr_matrix(
                (
                    PRICE_WEIGHT
                    *
                    price_z
                )[:, None]
            )
        ],
        format="csr"
    )


    # ========================================================
    # 6. L2 NORMALIZE
    # ========================================================

    X = normalize(
        X,
        norm="l2",
        copy=False
    )


    return X.tocsr()

✅ HashingVectorizer ready
✅ IDF ready: (262144,)


In [33]:
# ============================================================
# BUILD FULL ITEM MATRIX
# CHECKPOINT THEO SHARD
# ============================================================

con = duckdb.connect()


reader = con.execute(f"""
    SELECT

        item_id,
        name,
        price,

        category0,
        category1,
        category2,

        brand,
        condition

    FROM read_parquet(
        '{CATALOG_SOURCE.as_posix()}'
    )
""").fetch_record_batch(
    BATCH_SIZE
)


processed = 0


for shard_id, batch in enumerate(
    reader
):

    matrix_path = (
        SHARD_DIR
        /
        f"matrix_{shard_id:05d}.npz"
    )


    ids_path = (
        SHARD_DIR
        /
        f"item_ids_{shard_id:05d}.npy"
    )


    # ========================================================
    # CHECKPOINT VALIDATION
    # ========================================================

    reuse = False


    if (
        matrix_path.exists()
        and
        ids_path.exists()
        and
        matrix_path.stat().st_size > 0
        and
        ids_path.stat().st_size > 0
    ):

        try:

            old_X = sp.load_npz(
                matrix_path
            )


            old_ids = np.load(
                ids_path,
                mmap_mode="r",
                allow_pickle=False
            )


            expected_cols = (
                HASH_FEATURES
                + 1
            )


            if (
                old_X.shape[0]
                == batch.num_rows

                and

                old_X.shape[1]
                == expected_cols

                and

                len(old_ids)
                == batch.num_rows
            ):

                reuse = True


            del (
                old_X,
                old_ids
            )


        except Exception:

            reuse = False


    if reuse:

        processed += (
            batch.num_rows
        )


        print(
            f"✅ shard "
            f"{shard_id:05d} "
            f"SKIP | "
            f"{processed:,}"
            f"/"
            f"{FULL_ITEMS:,}"
        )


        continue


    # ========================================================
    # BUILD SHARD
    # ========================================================

    df_batch = (
        batch.to_pandas()
    )


    X = transform_metadata(
        df_batch
    )


    item_ids = np.asarray(

        df_batch[
            "item_id"
        ]
        .astype(str)
        .tolist(),

        dtype="U"
    )


    # ========================================================
    # ATOMIC SAVE
    # ========================================================

    temp_matrix = (
        SHARD_DIR
        /
        f"matrix_{shard_id:05d}.tmp.npz"
    )


    temp_ids = (
        SHARD_DIR
        /
        f"item_ids_{shard_id:05d}.tmp.npy"
    )


    sp.save_npz(
        temp_matrix,
        X,
        compressed=True
    )


    np.save(
        temp_ids,
        item_ids
    )


    temp_matrix.replace(
        matrix_path
    )


    temp_ids.replace(
        ids_path
    )


    processed += len(
        df_batch
    )


    print(
        f"💾 shard "
        f"{shard_id:05d}: "
        f"{len(df_batch):,} | "
        f"{processed:,}"
        f"/"
        f"{FULL_ITEMS:,}"
    )


    del (
        df_batch,
        X,
        item_ids
    )


    gc.collect()


con.close()


print()
print(
    "✅ MATRIX BUILD PASS COMPLETE"
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_5556\404636985.py:26: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(


💾 shard 00000: 100,000 | 100,000/30,044,194
💾 shard 00001: 100,000 | 200,000/30,044,194
💾 shard 00002: 100,000 | 300,000/30,044,194
💾 shard 00003: 100,000 | 400,000/30,044,194
💾 shard 00004: 100,000 | 500,000/30,044,194
💾 shard 00005: 100,000 | 600,000/30,044,194
💾 shard 00006: 100,000 | 700,000/30,044,194
💾 shard 00007: 100,000 | 800,000/30,044,194
💾 shard 00008: 100,000 | 900,000/30,044,194
💾 shard 00009: 100,000 | 1,000,000/30,044,194
💾 shard 00010: 100,000 | 1,100,000/30,044,194
💾 shard 00011: 100,000 | 1,200,000/30,044,194
💾 shard 00012: 100,000 | 1,300,000/30,044,194
💾 shard 00013: 100,000 | 1,400,000/30,044,194
💾 shard 00014: 100,000 | 1,500,000/30,044,194
💾 shard 00015: 100,000 | 1,600,000/30,044,194
💾 shard 00016: 100,000 | 1,700,000/30,044,194
💾 shard 00017: 100,000 | 1,800,000/30,044,194
💾 shard 00018: 100,000 | 1,900,000/30,044,194
💾 shard 00019: 100,000 | 2,000,000/30,044,194
💾 shard 00020: 100,000 | 2,100,000/30,044,194
💾 shard 00021: 100,000 | 2,200,000/30,044,194
💾 shar

In [34]:
# ============================================================
# MODEL MANIFEST
# ============================================================

matrix_files = sorted(
    SHARD_DIR.glob(
        "matrix_*.npz"
    )
)


manifest = []

saved_items = 0


for matrix_path in matrix_files:

    shard_id = int(
        matrix_path.stem
        .split("_")[1]
    )


    ids_path = (
        SHARD_DIR
        /
        f"item_ids_{shard_id:05d}.npy"
    )


    if not ids_path.exists():

        continue


    ids = np.load(
        ids_path,
        mmap_mode="r",
        allow_pickle=False
    )


    n_items = len(
        ids
    )


    saved_items += (
        n_items
    )


    manifest.append(
        {
            "shard":
                shard_id,

            "matrix":
                matrix_path.name,

            "item_ids":
                ids_path.name,

            "items":
                n_items,
        }
    )


MANIFEST_PATH.write_text(

    json.dumps(
        manifest,
        indent=2
    ),

    encoding="utf-8"
)


meta = {

    "model":
        "content_based_hashing_tfidf",

    "catalog":
        str(
            CATALOG_SOURCE
        ),

    "candidate_items":
        saved_items,

    "text_features":
        HASH_FEATURES,

    "matrix_features":
        HASH_FEATURES + 1,

    "price_weight":
        PRICE_WEIGHT,

    "price_log_mean":
        PRICE_LOG_MEAN,

    "price_log_std":
        PRICE_LOG_STD,

    "shards":
        len(
            manifest
        ),

    "metadata_scope":
        "train+val+test",

    "val_history":
        "train",

    "test_history":
        "train+val",
}


META_PATH.write_text(

    json.dumps(
        meta,
        indent=2
    ),

    encoding="utf-8"
)


print(
    "Saved items:",
    f"{saved_items:,}"
)


print(
    "Shards:",
    len(
        manifest
    )
)


if saved_items == FULL_ITEMS:

    print(
        "✅ FULL CATALOG COMPLETE"
    )

else:

    print(
        "⚠️ Chưa đủ catalog."
    )

    print(
        "Chạy lại Cell 9 "
        "để resume."
    )

Saved items: 30,044,194
Shards: 301
✅ FULL CATALOG COMPLETE


In [35]:
# ============================================================
# LOAD EVALUATION USERS + TARGETS
# ============================================================

def load_eval_users_targets(
    split,
    n_users
):

    split = (
        split.lower()
    )


    if split == "val":

        target_glob = (
            VAL_GLOB
        )


        history_sql = f"""
            SELECT DISTINCT
                CAST(
                    user_id
                    AS VARCHAR
                ) AS user_id

            FROM read_parquet(
                '{TRAIN_GLOB}'
            )

            WHERE
                user_id IS NOT NULL
        """


    elif split == "test":

        target_glob = (
            TEST_GLOB
        )


        history_sql = f"""
            SELECT DISTINCT
                CAST(
                    user_id
                    AS VARCHAR
                ) AS user_id

            FROM read_parquet(
                '{TRAIN_GLOB}'
            )

            WHERE
                user_id IS NOT NULL


            UNION


            SELECT DISTINCT
                CAST(
                    user_id
                    AS VARCHAR
                ) AS user_id

            FROM read_parquet(
                '{VAL_GLOB}'
            )

            WHERE
                user_id IS NOT NULL
        """


    else:

        raise ValueError(
            "split phải là "
            "'val' hoặc 'test'"
        )


    con = duckdb.connect()


    users = con.execute(f"""
        WITH history_users AS (

            {history_sql}

        )

        SELECT DISTINCT

            CAST(
                e.user_id
                AS VARCHAR
            ) AS user_id

        FROM read_parquet(
            '{target_glob}'
        ) e

        INNER JOIN history_users h

        ON
            CAST(
                e.user_id
                AS VARCHAR
            )
            =
            h.user_id

        WHERE
            e.user_id IS NOT NULL
            AND
            e.item_id IS NOT NULL

        ORDER BY
            hash(
                CAST(
                    e.user_id
                    AS VARCHAR
                )
            )

        LIMIT {int(n_users)}
    """).df()


    con.register(
        "eval_users",
        users
    )


    targets = con.execute(f"""
        SELECT DISTINCT

            CAST(
                e.user_id
                AS VARCHAR
            ) AS user_id,

            CAST(
                e.item_id
                AS VARCHAR
            ) AS item_id,

            e.event_group,

            e.is_strong_positive,

            e.is_purchase

        FROM read_parquet(
            '{target_glob}'
        ) e

        INNER JOIN eval_users u

        ON
            CAST(
                e.user_id
                AS VARCHAR
            )
            =
            u.user_id

        WHERE
            e.item_id IS NOT NULL
    """).df()


    con.register(
        "targets_tmp",
        targets
    )


    covered = con.execute(f"""
        SELECT DISTINCT

            t.user_id,
            t.item_id

        FROM targets_tmp t

        INNER JOIN read_parquet(
            '{CATALOG_SOURCE.as_posix()}'
        ) c

        ON
            t.item_id
            =
            CAST(
                c.item_id
                AS VARCHAR
            )
    """).df()


    con.close()


    coverage = (

        len(
            covered
        )

        /

        max(
            len(
                targets
            ),
            1
        )
    )


    print(
        "=" * 60
    )

    print(
        "SPLIT:",
        split.upper()
    )

    print(
        "Users:",
        f"{len(users):,}"
    )

    print(
        "Targets:",
        f"{len(targets):,}"
    )

    print(
        "Metadata coverage:",
        f"{coverage:.2%}"
    )

    print(
        "=" * 60
    )


    return (
        users,
        targets,
        coverage
    )

In [36]:
# ============================================================
# USER PROFILE
# ============================================================

def build_user_profiles(
    users,
    split,
    max_items=100
):

    split = (
        split.lower()
    )


    if split == "val":

        history_source = f"""
            SELECT *

            FROM read_parquet(
                '{TRAIN_GLOB}'
            )
        """


    elif split == "test":

        history_source = f"""
            SELECT *

            FROM read_parquet(
                '{TRAIN_GLOB}'
            )

            UNION ALL

            SELECT *

            FROM read_parquet(
                '{VAL_GLOB}'
            )
        """


    else:

        raise ValueError(
            "split phải là "
            "'val' hoặc 'test'"
        )


    con = duckdb.connect()


    con.register(
        "eval_users",
        users
    )


    history = con.execute(f"""
        WITH raw_history AS (

            SELECT

                CAST(
                    h.user_id
                    AS VARCHAR
                ) AS user_id,

                CAST(
                    h.item_id
                    AS VARCHAR
                ) AS item_id,

                SUM(
                    CAST(
                        h.event_weight
                        AS DOUBLE
                    )
                ) AS implicit_score,

                MAX(
                    h.ts
                ) AS last_ts

            FROM (

                {history_source}

            ) h

            INNER JOIN eval_users u

            ON
                CAST(
                    h.user_id
                    AS VARCHAR
                )
                =
                u.user_id

            WHERE
                h.item_id IS NOT NULL

            GROUP BY
                1,
                2
        ),

        ranked AS (

            SELECT

                *,

                ROW_NUMBER() OVER (

                    PARTITION BY
                        user_id

                    ORDER BY

                        last_ts DESC,

                        implicit_score DESC,

                        hash(
                            item_id
                        )
                ) AS rn

            FROM raw_history
        )

        SELECT

            h.user_id,
            h.item_id,
            h.implicit_score,
            h.last_ts,
            h.rn,

            c.name,
            c.price,

            c.category0,
            c.category1,
            c.category2,

            c.brand,
            c.condition

        FROM ranked h

        INNER JOIN read_parquet(
            '{CATALOG_SOURCE.as_posix()}'
        ) c

        ON
            h.item_id
            =
            CAST(
                c.item_id
                AS VARCHAR
            )

        WHERE
            h.rn
            <=
            {int(max_items)}

        ORDER BY
            h.user_id,
            h.rn
    """).df()


    seen_df = con.execute(f"""
        SELECT DISTINCT

            CAST(
                h.user_id
                AS VARCHAR
            ) AS user_id,

            CAST(
                h.item_id
                AS VARCHAR
            ) AS item_id

        FROM (

            {history_source}

        ) h

        INNER JOIN eval_users u

        ON
            CAST(
                h.user_id
                AS VARCHAR
            )
            =
            u.user_id

        WHERE
            h.item_id IS NOT NULL
    """).df()


    con.close()


    if history.empty:

        raise RuntimeError(
            "Không tạo được user history."
        )


    # ========================================================
    # ITEM VECTORS
    # ========================================================

    X = transform_metadata(
        history
    )


    user_ids = (
        users[
            "user_id"
        ]
        .astype(str)
        .tolist()
    )


    user_to_idx = {
        user_id:
        idx

        for idx, user_id
        in enumerate(
            user_ids
        )
    }


    row_users = (
        history[
            "user_id"
        ]
        .map(
            user_to_idx
        )
        .to_numpy()
    )


    # ========================================================
    # INTERACTION WEIGHT
    # ========================================================

    strength = np.log1p(

        np.maximum(

            history[
                "implicit_score"
            ]
            .to_numpy(
                dtype=np.float32
            ),

            0
        )
    )


    # ========================================================
    # RECENCY WEIGHT
    # rn bắt đầu từ 1
    # ========================================================

    recency = np.power(

        RECENCY_DECAY,

        (
            history[
                "rn"
            ]
            .to_numpy()
            -
            1
        )

    ).astype(
        np.float32
    )


    weights = (

        strength
        *
        recency

    ).astype(
        np.float32
    )


    W = sp.csr_matrix(

        (
            weights,

            (
                row_users,

                np.arange(
                    len(
                        history
                    )
                )
            )
        ),

        shape=(
            len(
                user_ids
            ),
            len(
                history
            )
        )
    )


    profiles = (
        W
        @
        X
    )


    profiles = normalize(
        profiles,
        norm="l2",
        copy=False
    ).tocsr()


    seen_dict = (

        seen_df

        .groupby(
            "user_id"
        )[
            "item_id"
        ]

        .agg(
            set
        )

        .to_dict()
    )


    print(
        "Profile rows:",
        f"{len(history):,}"
    )

    print(
        "Profiles:",
        profiles.shape
    )


    return (
        user_ids,
        profiles,
        seen_dict
    )

In [37]:
# ============================================================
# FULL CATALOG SEARCH
# ============================================================

def recommend_full_catalog(
    user_ids,
    profiles,
    seen_dict,
    k=20,
    query_batch=8
):

    shard_entries = []


    for matrix_path in sorted(
        SHARD_DIR.glob(
            "matrix_*.npz"
        )
    ):

        shard_id = int(
            matrix_path.stem
            .split("_")[1]
        )


        ids_path = (
            SHARD_DIR
            /
            f"item_ids_{shard_id:05d}.npy"
        )


        if ids_path.exists():

            shard_entries.append(
                (
                    matrix_path,
                    ids_path
                )
            )


    if not shard_entries:

        raise RuntimeError(
            "Chưa có matrix shards. "
            "Chạy Cell 9."
        )


    n_users = len(
        user_ids
    )


    best_items = [
        np.empty(
            0,
            dtype=object
        )

        for _
        in range(
            n_users
        )
    ]


    best_scores = [
        np.empty(
            0,
            dtype=np.float32
        )

        for _
        in range(
            n_users
        )
    ]


    print(
        "Shards:",
        len(
            shard_entries
        )
    )


    for shard_no, (
        matrix_path,
        ids_path
    ) in enumerate(
        shard_entries,
        1
    ):

        X_items = sp.load_npz(
            matrix_path
        ).tocsr()


        item_ids = np.load(
            ids_path,
            mmap_mode="r",
            allow_pickle=False
        )


        for start in range(
            0,
            n_users,
            query_batch
        ):

            end = min(
                start
                +
                query_batch,

                n_users
            )


            Q = profiles[
                start:end
            ]


            scores = (
                X_items
                @
                Q.T
            ).tocsc()


            for local_idx in range(
                end
                -
                start
            ):

                user_idx = (
                    start
                    +
                    local_idx
                )


                user_id = (
                    user_ids[
                        user_idx
                    ]
                )


                col_start = (
                    scores.indptr[
                        local_idx
                    ]
                )


                col_end = (
                    scores.indptr[
                        local_idx + 1
                    ]
                )


                rows = (
                    scores.indices[
                        col_start:
                        col_end
                    ]
                )


                vals = (
                    scores.data[
                        col_start:
                        col_end
                    ]
                )


                if len(
                    vals
                ) == 0:

                    continue


                seen = seen_dict.get(
                    user_id,
                    set()
                )


                # Lấy dư candidate trước khi filter seen.
                local_take = min(

                    len(
                        vals
                    ),

                    max(
                        200,
                        k
                        +
                        min(
                            len(seen),
                            1000
                        )
                    )
                )


                if (
                    len(vals)
                    >
                    local_take
                ):

                    pos = np.argpartition(
                        vals,
                        -local_take
                    )[
                        -local_take:
                    ]

                else:

                    pos = np.arange(
                        len(
                            vals
                        )
                    )


                pos = pos[
                    np.argsort(
                        vals[
                            pos
                        ]
                    )[::-1]
                ]


                candidate_items = (
                    item_ids[
                        rows[
                            pos
                        ]
                    ]
                )


                candidate_scores = (
                    vals[
                        pos
                    ]
                )


                new_items = []
                new_scores = []


                for item_id, score in zip(
                    candidate_items,
                    candidate_scores
                ):

                    item_id = str(
                        item_id
                    )


                    if item_id in seen:

                        continue


                    new_items.append(
                        item_id
                    )


                    new_scores.append(
                        float(
                            score
                        )
                    )


                    if (
                        len(
                            new_items
                        )
                        >=
                        k
                    ):

                        break


                if not new_items:

                    continue


                merged_items = np.concatenate(
                    [
                        best_items[
                            user_idx
                        ],

                        np.asarray(
                            new_items,
                            dtype=object
                        )
                    ]
                )


                merged_scores = np.concatenate(
                    [
                        best_scores[
                            user_idx
                        ],

                        np.asarray(
                            new_scores,
                            dtype=np.float32
                        )
                    ]
                )


                order = np.argsort(
                    merged_scores
                )[::-1]


                # dedupe item_id
                chosen_items = []
                chosen_scores = []
                used = set()


                for idx in order:

                    item = str(
                        merged_items[
                            idx
                        ]
                    )


                    if item in used:

                        continue


                    used.add(
                        item
                    )


                    chosen_items.append(
                        item
                    )


                    chosen_scores.append(
                        merged_scores[
                            idx
                        ]
                    )


                    if (
                        len(
                            chosen_items
                        )
                        >=
                        k
                    ):

                        break


                best_items[
                    user_idx
                ] = np.asarray(
                    chosen_items,
                    dtype=object
                )


                best_scores[
                    user_idx
                ] = np.asarray(
                    chosen_scores,
                    dtype=np.float32
                )


        if (
            shard_no % 10 == 0
            or
            shard_no
            ==
            len(
                shard_entries
            )
        ):

            print(
                f"✅ "
                f"{shard_no}"
                f"/"
                f"{len(shard_entries)} "
                f"shards"
            )


        del (
            X_items,
            item_ids
        )


        gc.collect()


    predictions = {}


    for i, user_id in enumerate(
        user_ids
    ):

        predictions[
            user_id
        ] = [

            str(
                x
            )

            for x
            in best_items[
                i
            ]
        ]


    return predictions

In [38]:
# ============================================================
# METRICS
# ============================================================

def make_target_dict(
    targets_df
):

    if targets_df.empty:

        return {}


    return (

        targets_df

        .groupby(
            "user_id"
        )[
            "item_id"
        ]

        .agg(
            set
        )

        .to_dict()
    )


def calculate_metrics(
    predictions,
    targets,
    k
):

    recalls = []

    precisions = []

    hitrates = []

    ndcgs = []


    for user_id, gt_items in (
        targets.items()
    ):

        if not gt_items:

            continue


        gt_items = {
            str(
                x
            )
            for x
            in gt_items
        }


        pred = (
            predictions
            .get(
                user_id,
                []
            )[:k]
        )


        hits = [

            1
            if item in gt_items
            else
            0

            for item
            in pred
        ]


        n_hits = sum(
            hits
        )


        recalls.append(

            n_hits

            /

            len(
                gt_items
            )
        )


        precisions.append(

            n_hits

            /

            k
        )


        hitrates.append(

            1.0
            if n_hits > 0
            else
            0.0
        )


        dcg = sum(

            rel

            /

            math.log2(
                rank + 2
            )

            for rank, rel
            in enumerate(
                hits
            )
        )


        ideal_hits = min(

            len(
                gt_items
            ),

            k
        )


        idcg = sum(

            1.0

            /

            math.log2(
                rank + 2
            )

            for rank
            in range(
                ideal_hits
            )
        )


        ndcgs.append(

            dcg
            /
            idcg

            if idcg > 0

            else 0.0
        )


    if not recalls:

        return {
            f"Recall@{k}":
                np.nan,

            f"Precision@{k}":
                np.nan,

            f"HitRate@{k}":
                np.nan,

            f"NDCG@{k}":
                np.nan,

            "users_eval":
                0
        }


    return {

        f"Recall@{k}":
            float(
                np.mean(
                    recalls
                )
            ),

        f"Precision@{k}":
            float(
                np.mean(
                    precisions
                )
            ),

        f"HitRate@{k}":
            float(
                np.mean(
                    hitrates
                )
            ),

        f"NDCG@{k}":
            float(
                np.mean(
                    ndcgs
                )
            ),

        "users_eval":
            len(
                recalls
            )
    }

In [39]:
# ============================================================
# VALIDATION
# ============================================================

(
    val_users,
    val_targets_df,
    val_metadata_coverage
) = load_eval_users_targets(
    split="val",
    n_users=EVAL_USERS_VAL
)


(
    val_user_ids,
    val_profiles,
    val_seen
) = build_user_profiles(
    users=val_users,
    split="val",
    max_items=PROFILE_MAX_ITEMS
)


val_predictions = recommend_full_catalog(
    user_ids=val_user_ids,
    profiles=val_profiles,
    seen_dict=val_seen,
    k=max(
        K_LIST
    ),
    query_batch=QUERY_BATCH
)


# ============================================================
# TARGET MODES
# ============================================================

val_all = (
    val_targets_df[
        [
            "user_id",
            "item_id"
        ]
    ]
    .drop_duplicates()
)


val_positive = (

    val_targets_df[

        val_targets_df[
            "event_group"
        ].isin(
            [
                "like",
                "cart",
                "offer",
                "buy_start",
                "buy_comp"
            ]
        )

    ][
        [
            "user_id",
            "item_id"
        ]
    ]

    .drop_duplicates()
)


val_strong = (

    val_targets_df[

        val_targets_df[
            "is_strong_positive"
        ]
        ==
        1

    ][
        [
            "user_id",
            "item_id"
        ]
    ]

    .drop_duplicates()
)


val_purchase = (

    val_targets_df[

        val_targets_df[
            "is_purchase"
        ]
        ==
        1

    ][
        [
            "user_id",
            "item_id"
        ]
    ]

    .drop_duplicates()
)


rows = []


for mode, target_df in [

    (
        "ALL",
        val_all
    ),

    (
        "POSITIVE",
        val_positive
    ),

    (
        "STRONG",
        val_strong
    ),

    (
        "PURCHASE",
        val_purchase
    )

]:

    target_dict = make_target_dict(
        target_df
    )


    row = {

        "split":
            "VAL",

        "target_mode":
            mode,

        "metadata_coverage":
            val_metadata_coverage,

        "targets":
            len(
                target_df
            ),
    }


    for k in K_LIST:

        row.update(

            calculate_metrics(
                val_predictions,
                target_dict,
                k
            )
        )


    rows.append(
        row
    )


val_result = pd.DataFrame(
    rows
)


display(
    val_result
)


val_result.to_csv(
    VAL_RESULT_PATH,
    index=False
)


print(
    "✅ saved:",
    VAL_RESULT_PATH
)

SPLIT: VAL
Users: 500
Targets: 6,858
Metadata coverage: 90.30%
Profile rows: 21,835
Profiles: (500, 262145)
Shards: 301
✅ 10/301 shards
✅ 20/301 shards
✅ 30/301 shards
✅ 40/301 shards
✅ 50/301 shards
✅ 60/301 shards
✅ 70/301 shards
✅ 80/301 shards
✅ 90/301 shards
✅ 100/301 shards
✅ 110/301 shards
✅ 120/301 shards
✅ 130/301 shards
✅ 140/301 shards
✅ 150/301 shards
✅ 160/301 shards
✅ 170/301 shards
✅ 180/301 shards
✅ 190/301 shards
✅ 200/301 shards
✅ 210/301 shards
✅ 220/301 shards
✅ 230/301 shards
✅ 240/301 shards
✅ 250/301 shards
✅ 260/301 shards
✅ 270/301 shards
✅ 280/301 shards
✅ 290/301 shards
✅ 300/301 shards
✅ 301/301 shards


,split,target_mode,metadata_coverage,targets,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,Recall@20,Precision@20,HitRate@20,NDCG@20
0,VAL,ALL,0.903033,6193,0.007261,0.002400,0.020000,0.003924,500,0.011157,0.002000,0.030000,0.005080
1,VAL,POSITIVE,0.903033,873,0.008673,0.001810,0.018100,0.004170,221,0.013348,0.001357,0.027149,0.005542
2,VAL,STRONG,0.903033,62,0.022222,0.002222,0.022222,0.011111,45,0.044444,0.002222,0.044444,0.017116
3,VAL,PURCHASE,0.903033,4,0.000000,0.000000,0.000000,0.000000,3,0.000000,0.000000,0.000000,0.000000


✅ saved: D:\MerRec\training\checkpoints\content_based_full_catalog\evaluation_val.csv


In [40]:
# ============================================================
# FINAL TEST
# ============================================================

(
    test_users,
    test_targets_df,
    test_metadata_coverage
) = load_eval_users_targets(
    split="test",
    n_users=EVAL_USERS_TEST
)


(
    test_user_ids,
    test_profiles,
    test_seen
) = build_user_profiles(
    users=test_users,
    split="test",
    max_items=PROFILE_MAX_ITEMS
)


test_predictions = recommend_full_catalog(
    user_ids=test_user_ids,
    profiles=test_profiles,
    seen_dict=test_seen,
    k=max(
        K_LIST
    ),
    query_batch=QUERY_BATCH
)


# ============================================================
# TARGET MODES
# ============================================================

test_all = (
    test_targets_df[
        [
            "user_id",
            "item_id"
        ]
    ]
    .drop_duplicates()
)


test_positive = (

    test_targets_df[

        test_targets_df[
            "event_group"
        ].isin(
            [
                "like",
                "cart",
                "offer",
                "buy_start",
                "buy_comp"
            ]
        )

    ][
        [
            "user_id",
            "item_id"
        ]
    ]

    .drop_duplicates()
)


test_strong = (

    test_targets_df[

        test_targets_df[
            "is_strong_positive"
        ]
        ==
        1

    ][
        [
            "user_id",
            "item_id"
        ]
    ]

    .drop_duplicates()
)


test_purchase = (

    test_targets_df[

        test_targets_df[
            "is_purchase"
        ]
        ==
        1

    ][
        [
            "user_id",
            "item_id"
        ]
    ]

    .drop_duplicates()
)


rows = []


for mode, target_df in [

    (
        "ALL",
        test_all
    ),

    (
        "POSITIVE",
        test_positive
    ),

    (
        "STRONG",
        test_strong
    ),

    (
        "PURCHASE",
        test_purchase
    )

]:

    target_dict = make_target_dict(
        target_df
    )


    row = {

        "split":
            "TEST",

        "target_mode":
            mode,

        "metadata_coverage":
            test_metadata_coverage,

        "targets":
            len(
                target_df
            ),
    }


    for k in K_LIST:

        row.update(

            calculate_metrics(
                test_predictions,
                target_dict,
                k
            )
        )


    rows.append(
        row
    )


test_result = pd.DataFrame(
    rows
)


display(
    test_result
)


test_result.to_csv(
    TEST_RESULT_PATH,
    index=False
)


print(
    "✅ saved:",
    TEST_RESULT_PATH
)

SPLIT: TEST
Users: 500
Targets: 5,859
Metadata coverage: 90.03%
Profile rows: 25,098
Profiles: (500, 262145)
Shards: 301
✅ 10/301 shards
✅ 20/301 shards
✅ 30/301 shards
✅ 40/301 shards
✅ 50/301 shards
✅ 60/301 shards
✅ 70/301 shards
✅ 80/301 shards
✅ 90/301 shards
✅ 100/301 shards
✅ 110/301 shards
✅ 120/301 shards
✅ 130/301 shards
✅ 140/301 shards
✅ 150/301 shards
✅ 160/301 shards
✅ 170/301 shards
✅ 180/301 shards
✅ 190/301 shards
✅ 200/301 shards
✅ 210/301 shards
✅ 220/301 shards
✅ 230/301 shards
✅ 240/301 shards
✅ 250/301 shards
✅ 260/301 shards
✅ 270/301 shards
✅ 280/301 shards
✅ 290/301 shards
✅ 300/301 shards
✅ 301/301 shards


,split,target_mode,metadata_coverage,targets,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,Recall@20,Precision@20,HitRate@20,NDCG@20
0,TEST,ALL,0.900324,5275,0.005449,0.001400,0.012000,0.003022,500,0.011347,0.001400,0.022000,0.004501
1,TEST,POSITIVE,0.900324,724,0.006510,0.001042,0.010417,0.002739,192,0.012514,0.001302,0.026042,0.004429
2,TEST,STRONG,0.900324,45,0.034483,0.003448,0.034483,0.013340,29,0.034483,0.001724,0.034483,0.013340
3,TEST,PURCHASE,0.900324,6,0.166667,0.016667,0.166667,0.064475,6,0.166667,0.008333,0.166667,0.064475


✅ saved: D:\MerRec\training\checkpoints\content_based_full_catalog\evaluation_test.csv


In [41]:
# ============================================================
# FINAL CHECK
# ============================================================

meta = json.loads(
    META_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "=" * 70
)

print(
    "CONTENT-BASED MODEL READY"
)

print(
    "=" * 70
)

print(
    "Catalog:",
    meta["catalog"]
)

print(
    "Candidate items:",
    f'{meta["candidate_items"]:,}'
)

print(
    "Text features:",
    f'{meta["text_features"]:,}'
)

print(
    "Matrix features:",
    f'{meta["matrix_features"]:,}'
)

print(
    "Shards:",
    meta["shards"]
)

print()

print(
    "Model folder:"
)

print(
    OUT
)

print()

print(
    "VAL result:"
)

display(
    val_result
)

print()

print(
    "TEST result:"
)

display(
    test_result
)

CONTENT-BASED MODEL READY
Catalog: D:\MerRec\data\processed\recommender\serving\item_catalog_full.parquet
Candidate items: 30,044,194
Text features: 262,144
Matrix features: 262,145
Shards: 301

Model folder:
D:\MerRec\training\checkpoints\content_based_full_catalog

VAL result:


,split,target_mode,metadata_coverage,targets,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,Recall@20,Precision@20,HitRate@20,NDCG@20
0,VAL,ALL,0.903033,6193,0.007261,0.002400,0.020000,0.003924,500,0.011157,0.002000,0.030000,0.005080
1,VAL,POSITIVE,0.903033,873,0.008673,0.001810,0.018100,0.004170,221,0.013348,0.001357,0.027149,0.005542
2,VAL,STRONG,0.903033,62,0.022222,0.002222,0.022222,0.011111,45,0.044444,0.002222,0.044444,0.017116
3,VAL,PURCHASE,0.903033,4,0.000000,0.000000,0.000000,0.000000,3,0.000000,0.000000,0.000000,0.000000



TEST result:


,split,target_mode,metadata_coverage,targets,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,Recall@20,Precision@20,HitRate@20,NDCG@20
0,TEST,ALL,0.900324,5275,0.005449,0.001400,0.012000,0.003022,500,0.011347,0.001400,0.022000,0.004501
1,TEST,POSITIVE,0.900324,724,0.006510,0.001042,0.010417,0.002739,192,0.012514,0.001302,0.026042,0.004429
2,TEST,STRONG,0.900324,45,0.034483,0.003448,0.034483,0.013340,29,0.034483,0.001724,0.034483,0.013340
3,TEST,PURCHASE,0.900324,6,0.166667,0.016667,0.166667,0.064475,6,0.166667,0.008333,0.166667,0.064475
